In [1]:
import pandas as pd

In [2]:
from pycaret.clustering import setup, create_model, assign_model, plot_model, save_model, load_model, predict_model

In [3]:
df = pd.read_csv('welcome_survey_simple_v2.csv', sep=';')
df.head()

,age,edu_level,fav_animals,fav_place,gender
0,<18,Podstawowe,Brak ulubionych,NaN,Kobieta
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna
2,45-54,Wyższe,Psy,W lesie,Mężczyzna
3,35-44,Średnie,Koty,W górach,Mężczyzna
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna


In [4]:
len(df)

229

In [5]:
s = setup(df, session_id=111)

,Description,Value
0,Session id,111
1,Original data shape,"(229, 5)"
2,Transformed data shape,"(229, 21)"
3,Categorical features,5
4,Rows with missing values,13.1%
5,Preprocess,True
6,Imputation type,simple
7,Numeric imputation,mean
8,Categorical imputation,mode
9,Maximum one-hot encoding,-1


In [6]:
s.dataset.head()

,age,edu_level,fav_animals,fav_place,gender
0,<18,Podstawowe,Brak ulubionych,NaN,Kobieta
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna
2,45-54,Wyższe,Psy,W lesie,Mężczyzna
3,35-44,Średnie,Koty,W górach,Mężczyzna
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna


In [7]:
s.dataset_transformed.head()

,age_<18,age_25-34,age_45-54,age_35-44,age_18-24,age_>=65,age_55-64,age_unknown,edu_level_Podstawowe,edu_level_Średnie,...,fav_animals_Brak ulubionych,fav_animals_Psy,fav_animals_Koty,fav_animals_Inne,fav_animals_Koty i Psy,fav_place_Nad wodą,fav_place_W lesie,fav_place_W górach,fav_place_Inne,gender
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [8]:
kmeans = create_model('kmeans', num_clusters=8)

,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness
0,0.1894,24.5274,1.7883,0,0,0


In [9]:
df_with_clusters = assign_model(kmeans)
df_with_clusters

,age,edu_level,fav_animals,fav_place,gender,Cluster
0,<18,Podstawowe,Brak ulubionych,NaN,Kobieta,Cluster 1
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna,Cluster 5
2,45-54,Wyższe,Psy,W lesie,Mężczyzna,Cluster 4
3,35-44,Średnie,Koty,W górach,Mężczyzna,Cluster 7
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna,Cluster 2
...,...,...,...,...,...,...
224,35-44,Wyższe,Koty,Inne,Kobieta,Cluster 6
225,45-54,Wyższe,Inne,W lesie,Mężczyzna,Cluster 4
226,25-34,Wyższe,Psy,W górach,Mężczyzna,Cluster 3
227,35-44,Wyższe,Brak ulubionych,W górach,Mężczyzna,Cluster 6


In [10]:
import json

cluster_descriptions = {}

for cluster_id in sorted(df_with_clusters["Cluster"].unique()):
    cluster_df = df_with_clusters[df_with_clusters["Cluster"] == cluster_id]

    top_gender = cluster_df["gender"].value_counts().idxmax()
    top_age = cluster_df["age"].value_counts().idxmax()
    top_place = cluster_df["fav_place"].value_counts().idxmax()
    top_animals = cluster_df["fav_animals"].value_counts().idxmax()
    top_edu = cluster_df["edu_level"].value_counts().idxmax()

    cluster_descriptions[cluster_id] = {
        "name": f"Grupa {cluster_id}",
        "description": (
            f"W tym klastrze znajdują się osoby, które najczęściej mają wykształcenie {top_edu}, "
            f"preferują spędzanie czasu {top_place.lower()}, "
            f"a ich ulubionymi zwierzętami są {top_animals.lower()}. "
            f"Większość z nich to {top_gender.lower()} w wieku {top_age}."
        )
    }

with open("welcome_survey_cluster_names_and_descriptions_v2.json", "w", encoding="utf-8") as f:
    json.dump(cluster_descriptions, f, ensure_ascii=False, indent=4)

cluster_descriptions


{'Cluster 0': {'name': 'Grupa Cluster 0',
  'description': 'W tym klastrze znajdują się osoby, które najczęściej mają wykształcenie Wyższe, preferują spędzanie czasu w lesie, a ich ulubionymi zwierzętami są koty. Większość z nich to mężczyzna w wieku 45-54.'},
 'Cluster 1': {'name': 'Grupa Cluster 1',
  'description': 'W tym klastrze znajdują się osoby, które najczęściej mają wykształcenie Wyższe, preferują spędzanie czasu nad wodą, a ich ulubionymi zwierzętami są psy. Większość z nich to mężczyzna w wieku 45-54.'},
 'Cluster 2': {'name': 'Grupa Cluster 2',
  'description': 'W tym klastrze znajdują się osoby, które najczęściej mają wykształcenie Wyższe, preferują spędzanie czasu nad wodą, a ich ulubionymi zwierzętami są psy. Większość z nich to mężczyzna w wieku 35-44.'},
 'Cluster 3': {'name': 'Grupa Cluster 3',
  'description': 'W tym klastrze znajdują się osoby, które najczęściej mają wykształcenie Wyższe, preferują spędzanie czasu w górach, a ich ulubionymi zwierzętami są psy. Więk

In [11]:
df_with_clusters["Cluster"].value_counts()

Cluster
Cluster 1    51
Cluster 2    37
Cluster 4    33
Cluster 6    30
Cluster 3    27
Cluster 7    20
Cluster 5    18
Cluster 0    13
Name: count, dtype: int64

In [12]:
plot_model(kmeans, plot='cluster')

In [13]:
save_model(kmeans, 'welcome_survey_simple_v2_pipeline', verbose=False)

(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=[], transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=['age', 'edu_level', 'fav_animals',
                                              'fav_place', 'gender'],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('ordinal_encoding',
                  TransformerWrapper(include=['gender'],
                                     transfo...
                                                                mapping=[{'col': 'gender',
                                                                          'data_type': dtype('O'),
                                                                          'mapping': Kobieta      0
 Mężczyzna    1
 NaN         -1
 dtype: int64}]))),
                 ('onehot_encoding',
                  TransformerWrappe

In [14]:
kmeans_pipeline = load_model('welcome_survey_simple_v2_pipeline')
kmeans_pipeline

Transformation Pipeline and Model Successfully Loaded


Pipeline(memory=FastMemory(location=C:\Users\patry\AppData\Local\Temp\joblib),
         steps=[('numerical_imputer',
                 TransformerWrapper(include=[], transformer=SimpleImputer())),
                ('categorical_imputer',
                 TransformerWrapper(include=['age', 'edu_level', 'fav_animals',
                                             'fav_place', 'gender'],
                                    transformer=SimpleImputer(strategy='most_frequent'))),
                ('ordinal_encoding',
                 Trans...
                                                               mapping=[{'col': 'gender',
                                                                         'data_type': dtype('O'),
                                                                         'mapping': Kobieta      0
Mężczyzna    1
NaN         -1
dtype: int64}]))),
                ('onehot_encoding',
                 TransformerWrapper(include=['age', 'edu_level', 'fav_animals',
                                             'fav_place'],
                                    transformer=OneHotEncoder(cols=['age',
                                                                    'edu_level',
                                                                    'fav_animals',
                                                                    'fav_place'],
                                                              handle_missing='return_nan',
                                                              use_cat_names=True))),
                ('trained_model', KMeans(random_state=111))])

In [3]:
df_check = pd.read_csv('welcome_survey_simple_v2.csv', sep=';')
df_check

,age,edu_level,fav_animals,fav_place,gender
0,<18,Podstawowe,Brak ulubionych,NaN,Kobieta
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna
2,45-54,Wyższe,Psy,W lesie,Mężczyzna
3,35-44,Średnie,Koty,W górach,Mężczyzna
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna
...,...,...,...,...,...
224,35-44,Wyższe,Koty,Inne,Kobieta
225,45-54,Wyższe,Inne,W lesie,Mężczyzna
226,25-34,Wyższe,Psy,W górach,Mężczyzna
227,35-44,Wyższe,Brak ulubionych,W górach,Mężczyzna


In [4]:
for col in df_check.columns:
    print(col)
    print(df_check[col].unique())
    print()


age
['<18' '25-34' '45-54' '35-44' '18-24' '>=65' '55-64' 'unknown']

edu_level
['Podstawowe' 'Średnie' 'Wyższe']

fav_animals
['Brak ulubionych' 'Psy' 'Koty' 'Inne' 'Koty i Psy']

fav_place
[nan 'Nad wodą' 'W lesie' 'W górach' 'Inne']

gender
['Kobieta' 'Mężczyzna' nan]

